# PDF Quality Audit
**CPIE — Week 2, Step 3**

Six checks per document. No pipeline code until every document has a confirmed action.

**Actions per document:** Include / Include with fixes / Replace with primary source / Drop from v1

In [ ]:
import sys
import re
from pathlib import Path
import fitz  # PyMuPDF
import pandas as pd
from IPython.display import display

RAW_DIR = Path('../data/raw')
pdfs = sorted(RAW_DIR.glob('*.pdf'))
print(f'Found {len(pdfs)} PDFs:')
for p in pdfs:
    print(f'  {p.name}')

## Institution mapping
Assign each PDF to institution and document type based on filename.

In [ ]:
# Manual mapping — update if filenames change
DOC_META = {
    '739682_ESO_Beyond2030_Report_2024_PRINT.pdf': {
        'institution': 'National Grid ESO',
        'doc_type': 'technical_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section',
    },
    'Progress-in-reducing-emissions-2024-Report-to-Parliament-Web.pdf': {
        'institution': 'CCC',
        'doc_type': 'statutory_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section',
    },
    'Progress-in-reducing-emissions-2025-report-to-Parliament.pdf': {
        'institution': 'CCC',
        'doc_type': 'statutory_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section',
    },
    'Smart-Secure-Electricity-Systems-Implementing-the-load-control-licensing-regime-consultation.pdf': {
        'institution': 'Ofgem',
        'doc_type': 'consultation',
        'jurisdiction': 'UK',
        'chunking_strategy': 'numbered_paragraph',
    },
    'The-Seventh-Carbon-Budget.pdf': {
        'institution': 'CCC',
        'doc_type': 'statutory_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section',
    },
    'WorldEnergyOutlook2025.pdf': {
        'institution': 'IEA',
        'doc_type': 'outlook_report',
        'jurisdiction': 'Global',
        'chunking_strategy': 'section_exec_summary_separate',
    },
    'boes-climate-related-financial-disclosure-2024.pdf': {
        'institution': 'BoE',
        'doc_type': 'disclosure_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section_heading',
    },
    'climate-change-possible-macroeconomic-implications.pdf': {
        'institution': 'BoE',
        'doc_type': 'discussion_paper',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section_heading',
    },
    'key-elements-2021-biennial-exploratory-scenario-financial-risks-climate-change.pdf': {
        'institution': 'FCA/PRA',
        'doc_type': 'scenario_analysis',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section_heading',
    },
    'measuring-climate-related-financial-risks-using-scenario-analysis.pdf': {
        'institution': 'FCA/PRA',
        'doc_type': 'methodology_paper',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section_heading',
    },
    'results-of-the-2021-climate-biennial-exploratory-scenario.pdf': {
        'institution': 'FCA/PRA',
        'doc_type': 'results_report',
        'jurisdiction': 'UK',
        'chunking_strategy': 'section_heading',
    },
    'zev-mandate-consultation-summary-of-responses-and-joint-government-response.pdf': {
        'institution': 'DESNZ',
        'doc_type': 'consultation_response',
        'jurisdiction': 'UK',
        'chunking_strategy': 'page_overlap',
    },
}
print('Mapping complete.')

## Check 1 — Extractability
Flag any document where >10% of pages return empty strings.

In [ ]:
def check_extractability(pdf_path: Path) -> dict:
    doc = fitz.open(str(pdf_path))
    n_pages = len(doc)
    empty_pages = []
    for i, page in enumerate(doc):
        text = page.get_text('text').strip()
        if not text:
            empty_pages.append(i + 1)
    doc.close()
    pct_empty = len(empty_pages) / n_pages if n_pages else 0
    return {
        'pages': n_pages,
        'empty_pages': len(empty_pages),
        'pct_empty': round(pct_empty, 3),
        'extractable': pct_empty <= 0.10,
        'empty_page_list': empty_pages[:10],  # first 10 for inspection
    }

extractability = {}
for pdf in pdfs:
    result = check_extractability(pdf)
    extractability[pdf.name] = result
    flag = '✓' if result['extractable'] else '✗ NEEDS OCR'
    print(f"{flag}  {pdf.name[:60]:<60} "
          f"{result['pages']} pages, {result['pct_empty']:.0%} empty")

## Check 2 — Layout quality
Spot-check 3 pages per document. Look for footnote/header bleed and coherence.

In [ ]:
def get_spot_pages(n_pages: int) -> list[int]:
    """Return 3 representative 0-indexed page numbers."""
    if n_pages <= 3:
        return list(range(n_pages))
    return [1, n_pages // 2, n_pages - 2]  # near start, middle, near end


def check_layout(pdf_path: Path, n_chars: int = 600) -> dict:
    doc = fitz.open(str(pdf_path))
    n_pages = len(doc)
    spot = get_spot_pages(n_pages)
    excerpts = {}
    for idx in spot:
        text = doc[idx].get_text('text').strip()
        excerpts[idx + 1] = text[:n_chars]  # page number (1-indexed)
    doc.close()
    return {'spot_pages': spot, 'excerpts': excerpts}


print("Spot-check excerpts (first 600 chars per page)\n" + "=" * 60)
layout_results = {}
for pdf in pdfs:
    result = check_layout(pdf)
    layout_results[pdf.name] = result
    print(f"\n{'─'*60}")
    print(f"FILE: {pdf.name}")
    for pg, text in result['excerpts'].items():
        print(f"  [Page {pg}]\n{text[:300]}\n")

### Layout quality assessment
After reading the excerpts above, fill in this dict manually.

In [ ]:
# Fill in after reading excerpts above.
# Values: True / False / 'partial'
LAYOUT_CLEAN = {
    '739682_ESO_Beyond2030_Report_2024_PRINT.pdf': None,
    'Progress-in-reducing-emissions-2024-Report-to-Parliament-Web.pdf': None,
    'Progress-in-reducing-emissions-2025-report-to-Parliament.pdf': None,
    'Smart-Secure-Electricity-Systems-Implementing-the-load-control-licensing-regime-consultation.pdf': None,
    'The-Seventh-Carbon-Budget.pdf': None,
    'WorldEnergyOutlook2025.pdf': None,
    'boes-climate-related-financial-disclosure-2024.pdf': None,
    'climate-change-possible-macroeconomic-implications.pdf': None,
    'key-elements-2021-biennial-exploratory-scenario-financial-risks-climate-change.pdf': None,
    'measuring-climate-related-financial-risks-using-scenario-analysis.pdf': None,
    'results-of-the-2021-climate-biennial-exploratory-scenario.pdf': None,
    'zev-mandate-consultation-summary-of-responses-and-joint-government-response.pdf': None,
}
print("Update LAYOUT_CLEAN after reading excerpts above.")

## Check 3 — Structural markers
Numbered paragraphs (Ofgem), section headings (FCA), chapter titles (DESNZ/IPCC/IEA).

In [ ]:
STRUCTURAL_PATTERNS = {
    'numbered_paragraph': re.compile(r'^\d+\.\d+', re.MULTILINE),   # 3.14, 4.1 etc.
    'section_heading': re.compile(r'^\d+\.?\s+[A-Z][A-Za-z ]{3,}', re.MULTILINE),
    'chapter_title': re.compile(r'^(Chapter|Section|Part)\s+\d+', re.MULTILINE | re.IGNORECASE),
    'executive_summary': re.compile(r'Executive Summary', re.IGNORECASE),
}


def check_structural_markers(pdf_path: Path) -> dict:
    doc = fitz.open(str(pdf_path))
    full_text = '\n'.join(page.get_text('text') for page in doc)
    doc.close()
    return {
        name: bool(pattern.search(full_text))
        for name, pattern in STRUCTURAL_PATTERNS.items()
    }


structural = {}
print(f"{'Document':<55} {'Num¶':<6} {'SecHdr':<8} {'Chapter':<9} {'ExecSum'}")
print('─' * 85)
for pdf in pdfs:
    r = check_structural_markers(pdf)
    structural[pdf.name] = r
    row = '  '.join('✓' if v else '✗' for v in r.values())
    print(f"{pdf.name[:54]:<55} {row}")

## Check 4 — Table detection
Detect pages with tables; assess whether extraction is coherent or garbage.

In [ ]:
def check_tables(pdf_path: Path) -> dict:
    doc = fitz.open(str(pdf_path))
    table_pages = []
    sample_table_text = None
    for i, page in enumerate(doc):
        tabs = page.find_tables()
        if tabs.tables:
            table_pages.append(i + 1)
            if sample_table_text is None:
                # Extract first table as plain text for coherence check
                try:
                    sample_table_text = tabs.tables[0].to_markdown()[:400]
                except Exception:
                    sample_table_text = 'extraction_failed'
    doc.close()
    return {
        'table_page_count': len(table_pages),
        'table_pages_sample': table_pages[:5],
        'sample_table': sample_table_text,
    }


table_results = {}
for pdf in pdfs:
    r = check_tables(pdf)
    table_results[pdf.name] = r
    has = r['table_page_count'] > 0
    print(f"{'⊞' if has else '○'}  {pdf.name[:60]:<60} {r['table_page_count']} table pages")

print("\n--- Sample table extractions (first table per doc with tables) ---")
for name, r in table_results.items():
    if r['sample_table']:
        print(f"\n{name}\n{r['sample_table']}")

## Check 5 — Relevance and worth
Four sub-questions per document. Fill in manually after reading Check 1–4 output.

In [ ]:
# Fill in after reviewing each document.
# queries_answerable: list of 3+ decision-framed queries this doc can answer (empty = fail)
# analyst_would_cite: True/False — would a climate finance analyst cite this?
# primary_source: True = this IS the primary source; False = a better source exists
# current_enough: True/False — content is actionable, not outdated

RELEVANCE = {
    '739682_ESO_Beyond2030_Report_2024_PRINT.pdf': {
        'queries_answerable': [
            # e.g. 'What grid investments does ESO recommend beyond 2030?',
        ],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'Progress-in-reducing-emissions-2024-Report-to-Parliament-Web.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'Progress-in-reducing-emissions-2025-report-to-Parliament.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'Smart-Secure-Electricity-Systems-Implementing-the-load-control-licensing-regime-consultation.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'The-Seventh-Carbon-Budget.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'WorldEnergyOutlook2025.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'boes-climate-related-financial-disclosure-2024.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'climate-change-possible-macroeconomic-implications.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'key-elements-2021-biennial-exploratory-scenario-financial-risks-climate-change.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'measuring-climate-related-financial-risks-using-scenario-analysis.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'results-of-the-2021-climate-biennial-exploratory-scenario.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
    'zev-mandate-consultation-summary-of-responses-and-joint-government-response.pdf': {
        'queries_answerable': [],
        'analyst_would_cite': None,
        'primary_source': None,
        'current_enough': None,
    },
}
print("Fill in RELEVANCE dict, then re-run this cell.")

## Check 6 — Embedding model + chunk size validation

Index 3 representative documents with:
- `all-MiniLM-L6-v2` (fast, 384-dim)
- `BAAI/bge-base-en-v1.5` (stronger, 768-dim)

Starting chunk size: **400 tokens / 80-token overlap**. Hard ceiling: 512 tokens.

Run 5 queries. Compare: top-5 chunk relevance (manual), cosine sim distribution, indexing time.

In [ ]:
import time
import tiktoken
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

TOKENIZER = tiktoken.get_encoding('cl100k_base')


def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))


def chunk_text(text: str, chunk_size: int = 400, overlap: int = 80) -> list[str]:
    tokens = TOKENIZER.encode(text)
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(tokens), step):
        chunk_tokens = tokens[start: start + chunk_size]
        if len(chunk_tokens) < 50:  # skip tiny trailing chunks
            break
        chunks.append(TOKENIZER.decode(chunk_tokens))
    return chunks


def extract_full_text(pdf_path: Path) -> str:
    doc = fitz.open(str(pdf_path))
    text = '\n'.join(page.get_text('text') for page in doc)
    doc.close()
    return text


# Pick 3 representative documents (Ofgem consultation, FCA paper, IEA report)
PILOT_DOCS = [
    RAW_DIR / 'Smart-Secure-Electricity-Systems-Implementing-the-load-control-licensing-regime-consultation.pdf',
    RAW_DIR / 'results-of-the-2021-climate-biennial-exploratory-scenario.pdf',
    RAW_DIR / 'WorldEnergyOutlook2025.pdf',
]

print('Extracting text from pilot docs...')
pilot_texts = {p.name: extract_full_text(p) for p in PILOT_DOCS if p.exists()}
print('Done.')

# Validate chunk size
print('\nChunk size validation (400 tokens / 80 overlap):')
for name, text in pilot_texts.items():
    chunks = chunk_text(text, chunk_size=400, overlap=80)
    token_counts = [count_tokens(c) for c in chunks]
    over_ceiling = sum(1 for t in token_counts if t > 512)
    print(f'  {name[:55]:<55} chunks={len(chunks):>4}  '
          f'avg={np.mean(token_counts):>5.0f}  '
          f'max={max(token_counts):>4}  '
          f'over512={over_ceiling}')

In [ ]:
# Build chunk corpus from pilot docs
pilot_chunks = []
for name, text in pilot_texts.items():
    for chunk in chunk_text(text):
        pilot_chunks.append({'source': name, 'text': chunk})

corpus = [c['text'] for c in pilot_chunks]
print(f'Total pilot chunks: {len(corpus)}')

In [ ]:
EVAL_QUERIES = [
    'What load control licensing requirements does Ofgem propose?',
    'How do climate transition risks affect bank balance sheets?',
    'What are the projected global energy demand trends to 2050?',
    'What stress test scenarios were used in the 2021 Climate BES?',
    'What is the expected cost of offshore wind capacity in the UK?',  # possible negative
]

MODELS = {
    'all-MiniLM-L6-v2': 'all-MiniLM-L6-v2',
    'bge-base-en-v1.5': 'BAAI/bge-base-en-v1.5',
}

model_results = {}

for label, model_name in MODELS.items():
    print(f'\nLoading {label}...')
    t0 = time.time()
    model = SentenceTransformer(model_name)
    load_time = time.time() - t0

    print(f'Encoding {len(corpus)} chunks...')
    t0 = time.time()
    corpus_embeddings = model.encode(corpus, show_progress_bar=True, batch_size=64)
    index_time = time.time() - t0

    query_embeddings = model.encode(EVAL_QUERIES)
    scores = cosine_similarity(query_embeddings, corpus_embeddings)  # (n_queries, n_chunks)

    model_results[label] = {
        'load_time': round(load_time, 1),
        'index_time': round(index_time, 1),
        'scores': scores,
        'dim': corpus_embeddings.shape[1],
    }
    print(f'  Load: {load_time:.1f}s  Index: {index_time:.1f}s  Dim: {corpus_embeddings.shape[1]}')

In [ ]:
TOP_K = 5

for label, res in model_results.items():
    print(f'\n{"="*70}')
    print(f'MODEL: {label}  (dim={res["dim"]}, index_time={res["index_time"]}s)')
    scores = res['scores']
    for q_idx, query in enumerate(EVAL_QUERIES):
        top_idx = scores[q_idx].argsort()[::-1][:TOP_K]
        print(f'\n  Q: {query}')
        for rank, ci in enumerate(top_idx, 1):
            src = pilot_chunks[ci]['source'][:40]
            snippet = pilot_chunks[ci]['text'][:120].replace('\n', ' ')
            print(f'    [{rank}] score={scores[q_idx][ci]:.3f}  src={src}')
            print(f'        {snippet}')

In [ ]:
# Cosine similarity distribution comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, res) in zip(axes, model_results.items()):
    all_scores = res['scores'].flatten()
    ax.hist(all_scores, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{label}\nindex={res["index_time"]}s  dim={res["dim"]}')
    ax.set_xlabel('Cosine similarity')
    ax.set_ylabel('Count')
    ax.axvline(np.mean(all_scores), color='red', linestyle='--', label=f'mean={np.mean(all_scores):.3f}')
    ax.legend()
plt.tight_layout()
plt.savefig('../data/eval/results/embedding_comparison.png', dpi=150)
plt.show()
print('Saved to data/eval/results/embedding_comparison.png')

### Lock embedding model decision
After inspecting top-5 chunks above, fill in this cell and update `configs/config.yaml`.

In [ ]:
# Fill in after inspection
LOCKED_EMBEDDING_MODEL = None  # 'all-MiniLM-L6-v2' or 'BAAI/bge-base-en-v1.5'
LOCKED_CHUNK_SIZE = None       # 400 unless validation showed otherwise
LOCKED_OVERLAP = None          # 80 unless validation showed otherwise

if LOCKED_EMBEDDING_MODEL:
    print(f'LOCKED: {LOCKED_EMBEDDING_MODEL}, chunk_size={LOCKED_CHUNK_SIZE}, overlap={LOCKED_OVERLAP}')
    print('Update configs/config.yaml with these values.')
else:
    print('Not yet locked — fill in values after inspection.')

## RRF k value test
Test k=10, 30, 60. Lock before building hybrid_retriever.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize_bm25(text: str) -> list[str]:
    return text.lower().split()


def rrf_score(bm25_rank: int, dense_rank: int, k: int) -> float:
    return 1 / (k + bm25_rank) + 1 / (k + dense_rank)


def run_rrf_comparison(query: str, corpus: list[str], dense_scores: np.ndarray, k_values: list[int]) -> dict:
    tokenized = [tokenize_bm25(c) for c in corpus]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = bm25.get_scores(tokenize_bm25(query))

    bm25_ranks = (-bm25_scores).argsort().argsort() + 1
    dense_ranks = (-dense_scores).argsort().argsort() + 1

    results = {}
    for k in k_values:
        rrf = np.array([rrf_score(bm25_ranks[i], dense_ranks[i], k) for i in range(len(corpus))])
        top5 = rrf.argsort()[::-1][:5]
        results[k] = top5
    return results


# Use the locked (or first available) model scores
model_label = list(model_results.keys())[0]
q_idx = 0  # first eval query
dense_scores_q = model_results[model_label]['scores'][q_idx]

rrf_results = run_rrf_comparison(EVAL_QUERIES[q_idx], corpus, dense_scores_q, k_values=[10, 30, 60])

print(f'RRF comparison for: "{EVAL_QUERIES[q_idx]}"\n')
for k, top_idx in rrf_results.items():
    print(f'k={k}:')
    for rank, ci in enumerate(top_idx, 1):
        snippet = corpus[ci][:100].replace('\n', ' ')
        print(f'  [{rank}] {snippet}')
    print()

In [ ]:
# Lock RRF k after inspection
LOCKED_RRF_K = None  # 10, 30, or 60

if LOCKED_RRF_K:
    print(f'LOCKED: RRF k={LOCKED_RRF_K}')
    print('Update configs/config.yaml with this value.')
else:
    print('Not yet locked — fill in after inspecting RRF results above.')

## Audit Summary Table
Run after completing all manual assessments (Checks 2, 5).

In [ ]:
def derive_action(ext: dict, layout_clean, structural: dict, relevance: dict) -> str:
    if not ext['extractable']:
        return 'Drop from v1 (not extractable)'
    if layout_clean is False:
        return 'Include with fixes'
    if relevance.get('analyst_would_cite') is False:
        return 'Drop from v1 (not relevant)'
    if relevance.get('primary_source') is False:
        return 'Replace with primary source'
    if relevance.get('current_enough') is False:
        return 'Drop from v1 (outdated)'
    if layout_clean is True and relevance.get('analyst_would_cite') is True:
        return 'Include'
    return 'Review needed'


rows = []
for pdf in pdfs:
    name = pdf.name
    ext = extractability.get(name, {})
    layout = LAYOUT_CLEAN.get(name)
    struct = structural.get(name, {})
    rel = RELEVANCE.get(name, {})
    tables = table_results.get(name, {})

    rows.append({
        'Document': name[:50],
        'Institution': DOC_META.get(name, {}).get('institution', '?'),
        'Pages': ext.get('pages', '?'),
        'Extractable': '✓' if ext.get('extractable') else '✗',
        'Layout clean': str(layout),
        'Struct markers': '✓' if any(struct.values()) else '✗',
        'Tables': tables.get('table_page_count', 0),
        'Relevant': str(rel.get('analyst_would_cite')),
        'Worth including': str(rel.get('primary_source')),
        'Action': derive_action(ext, layout, struct, rel),
    })

audit_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)
display(audit_df)

audit_df.to_csv('../data/eval/results/pdf_audit.csv', index=False)
print('\nSaved to data/eval/results/pdf_audit.csv')
print(f"\nAction summary:")
print(audit_df['Action'].value_counts().to_string())